<a href="https://colab.research.google.com/github/SamarthD07/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SamarthD07/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents one content item for one client on one report date (client_hash_id × content_hash_id × report_date). The feature window is February 2026 (2026-02-01 to 2026-02-28), and the label window is March 2026 (2026-03-01 to 2026-03-31). The two windows are kept separate so that features only use information available before the label period.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check the available files/tables and inspect the dataset structure
print("Date range:", feb["report_date"].min(), "to", feb["report_date"].max())

print("Rows:", len(feb))

print("Duplicate client-content-date rows:",
      feb.duplicated(["client_hash_id", "content_hash_id", "report_date"]).sum())

print("Unique client-content-date rows:",
      feb[["client_hash_id", "content_hash_id", "report_date"]].drop_duplicates().shape[0])

Date range: 2026-02-01 to 2026-02-28
Rows: 7355108
Duplicate client-content-date rows: 0
Unique client-content-date rows: 7355108


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: gsc_impressions, gsc_clicks, gsc_avg_position, ga4_pageviews, ga4_sessions, ga4_engaged_sessions, ga4_total_engagement_sec, and scroll_events. Label/proxy: a future March page-performance outcome used as a proxy for refresh priority. Context: client_hash_id, content_hash_id, report_date, client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available, and traffic-source fields such as sessions_organic, sessions_direct, sessions_referral, sessions_social, and sessions_paid. Excluded: AI-source fields such as sessions_ai, ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, and ai_other are excluded from the initial feature set because they are not necessary for the first version of the analysis and may contain sparse/zero-heavy signals.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("All columns:")
for i, col in enumerate(feb.columns, 1):
    print(i, col)

All columns:
1 report_date
2 client_hash_id
3 content_hash_id
4 client_has_gsc
5 client_has_ga4
6 gsc_data_available
7 ga4_data_available
8 gsc_impressions
9 gsc_clicks
10 gsc_sum_position
11 gsc_avg_position
12 ga4_pageviews
13 ga4_sessions
14 ga4_users
15 ga4_engaged_sessions
16 ga4_total_engagement_sec
17 sessions_organic
18 sessions_direct
19 sessions_referral
20 sessions_social
21 sessions_paid
22 sessions_ai
23 ai_chatgpt
24 ai_perplexity
25 ai_gemini
26 ai_copilot
27 ai_claude
28 ai_meta
29 ai_other
30 scroll_events


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
check_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "ga4_total_engagement_sec",
    "scroll_events"
]

print("Missing values in selected features:")
print(feb[check_cols].isna().sum())
print("Data availability counts:")

print("\nGSC available:")
print(feb["gsc_data_available"].value_counts(dropna=False))

print("\nGA4 available:")
print(feb["ga4_data_available"].value_counts(dropna=False))

print("\nGSC access:")
print(feb["client_has_gsc"].value_counts(dropna=False))

print("\nGA4 access:")
print(feb["client_has_ga4"].value_counts(dropna=False))
mar_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

mar = pd.read_parquet(mar_file)

print("March rows:", len(mar))
print("March date range:", mar["report_date"].min(), "to", mar["report_date"].max())
print("March columns:", len(mar.columns))
print("March numeric columns:")
print(mar.select_dtypes(include="number").columns.tolist())
print(
    mar[
        ["gsc_impressions",
         "gsc_clicks",
         "ga4_pageviews",
         "ga4_sessions",
         "ga4_engaged_sessions",
         "ga4_total_engagement_sec",
         "scroll_events"]
    ].describe().T
)

Missing values in selected features:
gsc_impressions               92256
gsc_clicks                    92256
gsc_avg_position            4733326
ga4_pageviews               4152323
ga4_sessions                4152323
ga4_engaged_sessions        4152323
ga4_total_engagement_sec    4152323
scroll_events               4152323
dtype: int64
Data availability counts:

GSC available:
gsc_data_available
False    4641069
True     2621783
None       92256
Name: count, dtype: int64

GA4 available:
ga4_data_available
None     4152323
False    3057464
True      145321
Name: count, dtype: int64

GSC access:
client_has_gsc
True     7262852
False      92256
Name: count, dtype: int64

GA4 access:
client_has_ga4
False    4476888
True     2878220
Name: count, dtype: int64
March rows: 9841378
March date range: 2026-03-01 to 2026-03-31
March columns: 30
March numeric columns:
['gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data cannot by itself prove the causal reason why a page's performance changed or guarantee that a page will improve after a refresh. The history is not perfectly balanced because GSC and GA4 availability differs across clients and dates. Early or partial rows may have GSC data without GA4 data, and feature and label windows must remain separate to avoid using future information. Therefore, the dataset supports observed performance and decision-support, not causal conclusions.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("February feature rows:", len(feb))
print("March label rows:", len(mar))

print("\nFebruary GSC availability:")
print(feb["gsc_data_available"].value_counts(dropna=False))

print("\nFebruary GA4 availability:")
print(feb["ga4_data_available"].value_counts(dropna=False))

print("\nFeature window:", feb["report_date"].min(), "to", feb["report_date"].max())
print("Label window:", mar["report_date"].min(), "to", mar["report_date"].max())

February feature rows: 7355108
March label rows: 9841378

February GSC availability:
gsc_data_available
False    4641069
True     2621783
None       92256
Name: count, dtype: int64

February GA4 availability:
ga4_data_available
None     4152323
False    3057464
True      145321
Name: count, dtype: int64

Feature window: 2026-02-01 to 2026-02-28
Label window: 2026-03-01 to 2026-03-31


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.